# Workflow Interface 1001: Workspace Creation from Jupyter Notebook

This tutorial demonstrates the methodology to convert a Federated Learning experiment developed in Jupyter Notebook into a Workspace that can be deployed using Aggregator Based Workflow

OpenFL experimental Workflow Interface enables the user to simulate a Federated Learning experiment using **LocalRuntime**. Once the simulation is ready, the methodology described in this tutorial enables the user to convert this experiment into an OpenFL workspace that can be deployed using the Aggregator-Based-Workflow

##### High Level Overview of Methodology
1. User annotates the relevant cells of the Jupyter notebook with `#| export` directive
2. We then Leverage `nbdev` functionality to export these annotated cells of Jupyter notebook into a Python script
3. Utilize OpenFL experimental module `WorkspaceExport` to convert the Python script into a OpenFL workspace
4. User can utilize the experimental `fx` commands to deploy and run the federation seamlessly


The methodology is described using an existing [OpenFL Watermarking Tutorial](https://github.com/securefederatedai/openfl/blob/develop/openfl-tutorials/experimental/Workflow_Interface_301_MNIST_Watermarking.ipynb). Let's get started !



# Getting Started

Initially, we start by specifying the module where cells marked with the `#| export` directive will be automatically exported. 

In the following cell, `#| default_exp experiment `indicates that the exported file will be named 'experiment'. This name can be modified based on user's requirement & preferences

In [1]:
#| default_exp experiment

Once we have specified the name of the module, subsequent cells of the notebook need to be *appended* by the `#| export` directive as shown below. User should ensure that *all* the notebook functionality required in the Federated Learning experiment is included in this directive

We start by installing OpenFL and dependencies of the workflow interface 
> These dependencies are required to be exported and become the requirements for the Federated Learning Workspace 

In [2]:
# #| export

# !pip install git+https://github.com/securefederatedai/openfl.git
# !pip install -r requirements_workflow_interface.txt

# !pip install matplotlib
# !pip install torch
# !pip install torchvision
# !pip install git+https://github.com/pyviz-topics/imagen.git@master
# !pip install holoviews==1.15.4


# # Uncomment this if running in Google Colab
# #!pip install -r https://raw.githubusercontent.com/intel/openfl/develop/openfl-tutorials/experimental/requirements_workflow_interface.txt
# #import os
# #os.environ["USERNAME"] = "colab"

We now define our dataloaders, model, optimizer, and some helper functions like we would for any other deep learning experiment 

> This cell and all the subsequent cells are important ingredients of the Federated Learning experiment and therefore annotated with the `#| export` directive

Next we import the `FLSpec`, placement decorators (`aggregator/collaborator`)

In [3]:
#| export

from openfl.experimental.interface.fl_spec import FLSpec
from openfl.experimental.placement.placement import aggregator, collaborator
import numpy as np


class bcolors:  # NOQA: N801
    HEADER = "\033[95m"
    OKBLUE = "\033[94m"
    OKCYAN = "\033[96m"
    OKGREEN = "\033[92m"
    WARNING = "\033[93m"
    FAIL = "\033[91m"
    ENDC = "\033[0m"
    BOLD = "\033[1m"
    UNDERLINE = "\033[4m"

/home/refaix/miniforge3/envs/dir_shift/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-11-06 19:02:49,128	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Let us now define the flow of internalloop testcase


In [4]:
#| export

class TestFlowInternalLoop(FLSpec):
    def __init__(self, model=None, optimizer=None, rounds= 3, **kwargs):
        super().__init__(**kwargs)
        self.training_rounds = rounds
        self.train_count = 0
        self.end_count = 0

    @aggregator
    def start(self):
        """
        Flow start.
        """
        print(
            f"{bcolors.OKBLUE}Testing FederatedFlow - "
            + f"Test for Internal Loops - Round: {self.train_count}"
            + f" of Training Rounds: {self.training_rounds}{bcolors.ENDC}"
        )
        self.model = np.zeros((10, 10, 10))  # Test model
        self.collaborators = self.runtime.collaborators
        self.next(self.agg_model_mean, foreach="collaborators")

    @collaborator
    def agg_model_mean(self):
        """
        Calculating the mean of the model created in start.
        """
        self.agg_mean_value = np.mean(self.model)
        print(f"<Collab>: {self.input} Mean of Agg model: {self.agg_mean_value} ")
        self.next(self.collab_model_update)

    @collaborator
    def collab_model_update(self):
        """
        Initializing the model with random numbers.
        """
        print(f"<Collab>: {self.input} Initializing the model randomly ")
        self.model = np.random.randint(1, len(self.input), (10, 10, 10))
        self.next(self.local_model_mean)

    @collaborator
    def local_model_mean(self):
        """
        Calculating the mean of the model created in train.
        """
        self.local_mean_value = np.mean(self.model)
        print(f"<Collab>: {self.input} Local mean: {self.local_mean_value} ")
        self.next(self.join)

    @aggregator
    def join(self, inputs):
        """
        Joining inputs from collaborators
        """
        self.agg_mean = sum(input.local_mean_value for input in inputs) / len(inputs)
        print(f"Aggregated mean : {self.agg_mean}")
        self.next(self.internal_loop)

    @aggregator
    def internal_loop(self):
        """
        Internally Loop for training rounds
        """
        self.train_count = self.train_count + 1
        if self.training_rounds == self.train_count:
            self.next(self.end)
        else:
            self.next(self.start)

    @aggregator
    def end(self):
        """
        This is the 'end' step. All flows must have an 'end' step, which is the
        last step in the flow.
        """
        self.end_count += 1
        print("This is the end of the flow")

        flflow = self
        # Flow Test Begins
        expected_flow_steps = [
            "join",
            "internal_loop",
            "agg_model_mean",
            "collab_model_update",
            "local_model_mean",
            "start",
        ]  # List to verify expected steps
        try:
            validate_flow(
                flflow, expected_flow_steps
            )  # Function to validate the internal flow
        except Exception as e:
            raise e
        # Flow Test Ends


def validate_flow(flow_obj, expected_flow_steps):
    """
    Validate:
    1. If the given training round were completed
    2. If all the steps were executed
    3. If each collaborator step was executed
    4. If end was executed once
    """
    validate_flow_error = []  # List to capture any errors in the flow

    from metaflow import Flow

    cli_flow_obj = Flow("TestFlowInternalLoop")  # Flow object from CLI
    cli_flow_steps = list(cli_flow_obj.latest_run)  # Steps from CLI
    cli_step_names = [step.id for step in cli_flow_steps]

    # 1. If the given training round were completed
    if not flow_obj.training_rounds == flow_obj.train_count:
        validate_flow_error.append(
            f"{bcolors.FAIL}... Error : Number of training completed is not equal"
            + f" to training rounds {bcolors.ENDC} \n"
        )

    for step in cli_flow_steps:
        task_count = 0
        func = getattr(flow_obj, step.id)
        for task in list(step):
            task_count = task_count + 1

        # Each aggregator step should be executed for training rounds times
        if (
            (func.aggregator_step is True)
            and (task_count != flow_obj.training_rounds)
            and (step.id != "end")
        ):
            validate_flow_error.append(
                f"{bcolors.FAIL}... Error : More than one execution detected for "
                + f"Aggregator Step: {step} {bcolors.ENDC} \n"
            )

        # Each collaborator step is executed for (training rounds)*(number of collaborator) times
        if (func.collaborator_step is True) and (
            task_count != len(flow_obj.collaborators) * flow_obj.training_rounds
        ):
            validate_flow_error.append(
                f"{bcolors.FAIL}... Error : Incorrect number of execution detected for "
                + f"Collaborator Step: {step}. Expected: "
                + f"{flow_obj.training_rounds*len(flow_obj.collaborators)} "
                + f"Actual: {task_count}{bcolors.ENDC} \n"
            )

    steps_present_in_cli = [
        step for step in expected_flow_steps if step in cli_step_names
    ]
    missing_steps_in_cli = [
        step for step in expected_flow_steps if step not in cli_step_names
    ]
    extra_steps_in_cli = [
        step for step in cli_step_names if step not in expected_flow_steps
    ]

    if len(steps_present_in_cli) != len(expected_flow_steps):
        validate_flow_error.append(
            f"{bcolors.FAIL}... Error : Number of steps fetched from Datastore through CLI do not "
            + f"match the Expected steps provided {bcolors.ENDC}  \n"
        )

    if len(missing_steps_in_cli) != 0:
        validate_flow_error.append(
            f"{bcolors.FAIL}... Error : Following steps missing from Datastore: "
            + f"{missing_steps_in_cli} {bcolors.ENDC}  \n"
        )

    if len(extra_steps_in_cli) != 0:
        validate_flow_error.append(
            f"{bcolors.FAIL}... Error : Following steps are extra in Datastore: "
            + f"{extra_steps_in_cli} {bcolors.ENDC}  \n"
        )

    if not flow_obj.end_count == 1:
        validate_flow_error.append(
            f"{bcolors.FAIL}... Error : End function called more than one time...{bcolors.ENDC}"
        )

    if validate_flow_error:
        display_validate_errors(validate_flow_error)
        raise Exception(f"{bcolors.FAIL}Test for Internal Loop FAILED")
    else:
        print(
            f"""{bcolors.OKGREEN}\n **** Summary of internal flow testing ****
        No issues found and below are the tests that ran successfully
        1. Number of training completed is equal to training rounds
        2. Cli steps and Expected steps are matching
        3. Number of tasks are aligned with number of rounds and number of collaborators
        4. End function executed one time {bcolors.ENDC}"""
        )


def display_validate_errors(validate_flow_error):
    """
    Function to display error that is captured during flow test
    """
    print("".join(validate_flow_error))


Aggregator step "start" registered
Collaborator step "agg_model_mean" registered
Collaborator step "collab_model_update" registered
Collaborator step "local_model_mean" registered
Aggregator step "join" registered
Aggregator step "internal_loop" registered
Aggregator step "end" registered




> NOTE: Aggregator based workflow requires a `FederatedRuntime`. In this methodology `FederatedRuntime` is created automatically and it's usage is transparent to the user

Now that we have our flow and runtime defined, let's run the experiment! 

## Workspace creation

The following cells convert the Jupyter notebook into a Python script and create a Template Workspace that can be utilized by Aggregator based Workflow
> NOTE: Only Notebook cells that were marked with `#| export` directive shall be included in this Python script

We first import `WorkspaceExport` module and execute `WorkspaceExport.export()` that converts the notebook and generates the template workspace. User is required to specify: 
1. `notebook_path`: path of the Jupyter notebook that is required to be converted
2. `output_workspace`: path where the converted workspace is stored

In [5]:
#| export

from openfl.experimental.runtime import FederatedRuntime

director_info = {
    'director_node_fqdn':'localhost',
    'director_port':50050,
    'cert_chain': None,
    'api_cert': None,
    'api_private_key': None,
}

# TODO: Is there a way to get the notebook path without passing it?
federated_runtime = FederatedRuntime(collaborators= ['env1','env2'], director=director_info, notebook_path='./testflow_internal_loop.ipynb')

In [6]:
federated_runtime.get_envoys()

['env1', 'env2']

In [7]:
#| export

flflow = TestFlowInternalLoop(checkpoint=True)
flflow.runtime = federated_runtime


In [8]:
flflow.run()


New experimental workspace directory structure:
generated_workspace
├── src
│   ├── __pycache__
│   ├── experiment.py
│   └── __init__.py
├── .workspace
├── plan
│   ├── defaults
│   ├── cols.yaml
│   ├── plan.yaml
│   └── data.yaml
└── requirements.txt

3 directories, 8 files
Aggregator step "start" registered
Collaborator step "agg_model_mean" registered
Collaborator step "collab_model_update" registered
Collaborator step "local_model_mean" registered
Aggregator step "join" registered
Aggregator step "internal_loop" registered
Aggregator step "end" registered
Archive created at /home/refaix/openfl_dir_new/openfl/openfl-tutorials/experimental/interactive_api/testflow_internalloop/workspace/experiment.zip
Experiment was submitted to the director!
Aggregator step "start" registered
Collaborator step "agg_model_mean" registered
Collaborator step "collab_model_update" registered
Collaborator step "local_model_mean" registered
Aggregator step "join" registered
Aggregator step "internal_lo

In [9]:
vars(flflow)

{'_foreach_methods': ['agg_model_mean',
  'join',
  'local_model_mean',
  'collab_model_update'],
 '_checkpoint': True,
 'training_rounds': 3,
 'train_count': 3,
 'end_count': 1,
 '_runtime': FederatedRuntime,
 'model': array([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],
 
        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0.,

## Workspace Usage

The workspace created above can be used by the Aggregator based workflow by using the `fx` commands in the following manner

**Workspace Activation and Creation**
1. Activate the experimental aggregator-based workflow:

    `fx experimental activate`

   This will create an 'experimental' directory under ~/.openfl/
3. Create a workspace using the custom template:

    `fx workspace create --prefix workspace_path --custom_template /home/$USER/generated-workspace`
4. Change to the workspace directory:

    `cd workspace_path`

**Workspace Initialization and Certification**
1. Initialize the FL plan and auto-populate the fully qualified domain name (FQDN) of the aggregator node:

    `fx plan initialize`
2. Certify the workspace:

    `fx workspace certify`
    
**Aggregator Setup and Workspace Export**
1. Run the aggregator certificate creation command:

    `fx aggregator generate-cert-request`

    `fx aggregator certify`
2. Export the workspace for collaboration:

    `fx workspace export`
    
**Collaborator Node Setup**

***On the Collaborator Node:***

1. Copy the workspace archive from the aggregator node to the collaborator nodes. Import the workspace archive:

    `fx workspace import --archive WORKSPACE.zip`
   
    `cd workspace_path`
3. Generate a collaborator certificate request:

    `fx collaborator generate-cert-request -n {COL_LABEL}`

***On the Aggregator Node (Certificate Authority):***

3. Sign the Collaborator Certificate Signing Request (CSR) Package from collaborator nodes:

    `fx collaborator certify --request-pkg /PATH/TO/col_{COL_LABEL}_to_agg_cert_request.zip`

***On the Collaborator Node:***

4. Import the signed certificate and certificate chain into the workspace:

    `fx collaborator certify --import /PATH/TO/agg_to_col_{COL_LABEL}_signed_cert.zip`
    
**Final Workspace Activation**
***On the Aggregator Node:***

1. Start the Aggregator:

    `fx aggregator start`
    
    The Aggregator is now running and waiting for Collaborators to connect.

***On the Collaborator Nodes:***

2. Run the Collaborator:

    `fx collaborator start -n {COL_LABEL}`

**Workspace Deactivation**
1. To deactivate the experimental aggregator-based workflow and switch back to original aggregator-based workflow:

    `fx experimental deactivate`

   This will remove the 'experimental' directory under ~/.openfl/
